# Spectrogram-Based Vishing Detector

## Imports and Configurations

In [1]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import math

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import (
    WaveformDataset,
    extract_archives,
    prepare_all_split_dataframes,
    subsample_by_class,
 )
from modules.attacks import FGSMAttack, FGSMAttackConfig
from modules.audio_processing import LogMelSpectrogramConfig
from modules.models import SpectrogramCNNConfig, SpectrogramClassifier

KeyboardInterrupt: 

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = MODEL_DIR / 'spectrogram'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
BONAFIDE_SUBSAMPLE = 1200
SPOOF_SUBSAMPLE = 1800
TARGET_NUM_SAMPLES = 64000
BATCH_SIZE = 128
NUM_EPOCHS = 70
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
RANDOM_SEED = 67
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EARLY_STOPPING_PATIENCE = 15

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT)})

## Data Preprocessing

In [ ]:
extract_archives(DATA_ROOT)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

In [ ]:
for split_name, frame in split_frames.items():
    counts = frame['LABEL'].value_counts().to_dict()
    total = sum(counts.values())
    print(f"{split_name}: {counts} | bonafide ratio: {counts.get(0,0)/total:.3f}")

In [ ]:
train_dataset = WaveformDataset(
    split_frames['train'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=True,
)
dev_dataset = WaveformDataset(
    split_frames['dev'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)
eval_dataset = WaveformDataset(
    split_frames['eval'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)

# Change all DataLoaders from num_workers=0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

## Training 

In [ ]:
frontend_config = LogMelSpectrogramConfig(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=80, f_min=20.0, f_max=7600.0)
model_config = SpectrogramCNNConfig(
    num_classes=2,
    base_channels=32,   # was 128 → gives 32→64→128→256 channels
    dropout=0.3,        # was 0.4
    embedding_dim=256   # was 512
)
model = SpectrogramClassifier(
    transformer_config=frontend_config,
    model_config=model_config,
    use_spec_augment=True,
).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_bonafide = (split_frames['train']['LABEL'] == 0).sum()
num_spoof = (split_frames['train']['LABEL'] == 1).sum()
ratio = num_spoof / num_bonafide  # true data should be ~9.0
class_weights = torch.tensor([ratio, 1.0], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.0
)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
model

In [ ]:
def run_epoch(model, dataloader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_examples = 0

    for batch in dataloader:
        waveforms = batch['waveform'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(waveforms)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ADD THIS
            optimizer.step()

        batch_size = labels.shape[0]
        total_examples += batch_size
        total_loss += loss.item() * batch_size

    return total_loss / max(total_examples, 1)

best_dev_loss = float('inf')
best_epoch = 0
epochs_without_improvement = 0  # ADD

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    dev_loss = run_epoch(model, dev_loader, optimizer=None)
    
    scheduler.step(dev_loss)
    
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_epoch = epoch
        epochs_without_improvement = 0  # ADD
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dev_loss': best_dev_loss,
        }, CHECKPOINT_DIR / "best.pt")
        
        print(f"→ New best model saved at epoch {epoch} (dev_loss: {dev_loss:.5f})")
    else:
        epochs_without_improvement += 1  # ADD
    
    print({
        'epoch': epoch, 
        'train_loss': round(train_loss, 5), 
        'dev_loss': round(dev_loss, 5),
        'best_dev_loss': round(best_dev_loss, 5),
        'best_epoch': best_epoch,
        'no_improve': epochs_without_improvement,  # ADD
    })

    # ADD early stopping block:
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch} (best epoch: {best_epoch})")
        break

## Evaluation

In [ ]:
fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=0.002,
        clamp_min=-1.0,
        clamp_max=1.0,
        targeted=False,
    )
)

all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for batch in eval_loader:
    waveforms = batch['waveform'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    # Obtaining clean predictions
    with torch.no_grad():
        clean_logits = model(waveforms)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_true_probabilities = clean_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    # Obtaining adversarial predictions
    attack_result = fgsm_attack.generate(model=model, inputs=waveforms, labels=labels)
    adversarial_waveforms = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_waveforms)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_true_probabilities = adversarial_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_true_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_true_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

### Accuracy

In [ ]:
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

### Mean Squared Confidence Degradation (MSCD)

In [ ]:
delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

### Attack Success Rate (ASR)

In [ ]:
asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"ASR: {asr:.4f}")